# Bài 47: Vision Transformer (ViT)

Bộ mã hóa Transformer ở Bài 46 hoạt động trên một chuỗi vector. Một ảnh không phải là một chuỗi — nên **Vision Transformer** ("An Image is Worth 16x16 Words", <a href="../references.html#dosovitskiy-2020">Dosovitskiy et al., 2020</a><span class="landmark-paper">&#9733;</span>) biến nó thành một chuỗi: cắt ảnh thành các mảng (patch) kích thước cố định, làm phẳng và nhúng tuyến tính mỗi mảng thành một vector, rồi đưa chuỗi thu được thẳng vào một bộ mã hóa Transformer thông thường, đúng như đã xây dựng ở Bài 46. Không có tích chập ở đâu cả. Bài này xây dựng quy trình đó, rồi chạy chính xác phép thử tổng quát hóa tịnh tiến ở Bài 34 để cho thấy chính xác một Transformer từ bỏ những gì khi bỏ đi thiên kiến quy nạp có sẵn của tích chập.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Chia mảng: biến một ảnh thành một chuỗi

Chia một ảnh `16x16` thành các mảng `4x4`, cho ra một chuỗi 16 mảng, mỗi mảng được làm phẳng thành một vector 16 số. Điều này chính xác tương đương với một phép tích chập có bước nhảy với kernel cùng kích thước với bước nhảy (`nn.Conv2d(..., kernel_size=P, stride=P)`) — cả hai đều trượt một cửa sổ không chồng lấn và áp dụng cùng một ánh xạ tuyến tính lên mỗi cửa sổ — đó là cách phép nhúng mảng của ViT thường được cài đặt trong thực tế.

In [ ]:
def patchify(img, patch_size):
    B, C, H, W = img.shape
    P = patch_size
    patches = img.unfold(2, P, P).unfold(3, P, P)  # (B, C, H/P, W/P, P, P)
    patches = patches.contiguous().view(B, C, -1, P, P).permute(0, 2, 1, 3, 4)
    return patches.reshape(B, -1, C * P * P)  # (B, num_patches, C*P*P)

img = torch.randn(2, 1, 16, 16)
patches = patchify(img, patch_size=4)
print(f'image {tuple(img.shape)} -> patch sequence {tuple(patches.shape)}  '
      f'({(16 // 4) ** 2} patches of {1 * 4 * 4} values each)')

patch_dim, embed_dim = 16, 8
linear_embed = nn.Linear(patch_dim, embed_dim)
conv_embed = nn.Conv2d(1, embed_dim, kernel_size=4, stride=4)
with torch.no_grad():
    conv_embed.weight.copy_(linear_embed.weight.view(embed_dim, 1, 4, 4))
    conv_embed.bias.copy_(linear_embed.bias)

out_linear = linear_embed(patches)
out_conv = conv_embed(img).flatten(2).transpose(1, 2)
print(f'linear-on-patches vs. strided-conv max abs diff: {(out_linear - out_conv).abs().max().item():.2e}')

## Token [CLS], và lắp ráp một ViT đầy đủ

Để biến một chuỗi các nhúng mảng thành một dự đoán duy nhất cho cả ảnh, ViT chèn thêm một token "phân loại" học được vào đầu chuỗi trước khi chạy bộ mã hóa Transformer (Bài 46) — sau khi chú ý đã cho nó thu thập thông tin từ mọi mảng, đầu ra cuối cùng của token đó chính là thứ mà phần đầu phân loại đọc. Mã hóa vị trí (Bài 46) được cộng vào để thứ tự mảng (tức *vị trí* mảng) không vô hình đối với cơ chế chú ý.

In [ ]:
def positional_encoding(T, D):
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(D).unsqueeze(0).float()
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / D))
    angles = pos * angle_rates
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe

class TinyViT(nn.Module):
    def __init__(self, img_size=16, patch_size=4, in_ch=1, embed_dim=32, n_heads=4, n_layers=2):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2
        patch_dim = in_ch * patch_size * patch_size
        self.embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.register_buffer('pos_enc', positional_encoding(n_patches + 1, embed_dim))
        layer = nn.TransformerEncoderLayer(embed_dim, n_heads, dim_feedforward=embed_dim * 2,
                                            batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(embed_dim, 1)

    def forward(self, x):
        p = patchify(x, self.patch_size)
        tok = self.embed(p)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        tok = torch.cat([cls, tok], dim=1) + self.pos_enc
        out = self.encoder(tok)
        return self.head(out[:, 0]).squeeze(-1)  # classify from the [CLS] token's output

model = TinyViT()
x = torch.randn(3, 1, 16, 16)
out = model(x)
print(f'TinyViT: input {tuple(x.shape)} -> output {tuple(out.shape)}  (one logit per image)')

## Thiên kiến quy nạp của tích chập mang lại gì cho bạn

Bài 34 đã huấn luyện một CNN và một MLP dựa trên làm phẳng trên các hình dấu-cộng-và-hình-tròn ở một khoảng vị trí, rồi kiểm tra cả hai ở các vị trí *chưa thấy* — CNN tổng quát hóa được (phép gộp cực đại toàn cục làm nó bất biến tịnh tiến chính xác); MLP dựa trên làm phẳng thì không (trọng số lớp đầu tiên của nó bị gắn với tọa độ điểm ảnh tuyệt đối). Hãy chạy lại chính xác thí nghiệm đó, thay `TinyViT` vào chỗ MLP.

Một ViT cũng không có tính bất biến tịnh tiến có sẵn — cả phép nhúng mảng lẫn mã hóa vị trí đều đặc thù theo vị trí, giống như đầu vào đã làm phẳng của MLP. Cơ chế thực tế chính mà ViT dùng để bù đắp là quy mô: tiền huấn luyện trên các tập dữ liệu khổng lồ cho phép cơ chế chú ý *học* được thứ gì đó giống tính bất biến tịnh tiến từ dữ liệu, thay vì nhận được nó miễn phí từ kiến trúc như CNN. Trên tập dữ liệu nhỏ bé trong khóa học này, hãy kỳ vọng việc học đó thất bại.

In [ ]:
def make_image(shape_type, cx, cy, size=16):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0.0 if shape_type == 'plus' else 1.0)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.float32)

data_rng = np.random.default_rng(1)
X_train, y_train = make_dataset(data_rng, 300, (5, 11))   # training positions
X_test, y_test = make_dataset(data_rng, 150, (3, 5))       # UNSEEN positions

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1))
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1)).squeeze(-1)

def train_and_eval(model_cls, Xtr, ytr, Xte, yte, seed, epochs=200, lr=0.001):
    torch.manual_seed(seed)
    model = model_cls()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        train_acc = ((model(Xtr) > 0).float() == ytr).float().mean().item()
        test_acc = ((model(Xte) > 0).float() == yte).float().mean().item()
    return train_acc, test_acc

Xtr_t = torch.tensor(X_train).unsqueeze(1); ytr_t = torch.tensor(y_train)
Xte_t = torch.tensor(X_test).unsqueeze(1); yte_t = torch.tensor(y_test)

for seed in range(3):
    vit_train, vit_test = train_and_eval(TinyViT, Xtr_t, ytr_t, Xte_t, yte_t, seed=seed)
    cnn_train, cnn_test = train_and_eval(CNNClassifier, Xtr_t, ytr_t, Xte_t, yte_t, seed=seed)
    print(f'seed={seed}: ViT train={vit_train:.1%} test(unseen positions)={vit_test:.1%}   '
          f'|   CNN train={cnn_train:.1%} test(unseen positions)={cnn_test:.1%}')

Cả hai mô hình đều khớp hoàn hảo các vị trí huấn luyện, nhưng ViT sụp đổ về mức đoán ngẫu nhiên (hoặc tệ hơn) trên các vị trí chưa thấy lần nào cũng vậy, trong khi CNN tổng quát hóa hoàn hảo — tái tạo gần như chính xác khoảng cách MLP-và-CNN ở Bài 34, với ViT đóng vai MLP. Đây là phát hiện thực nghiệm đã được ghi nhận rõ đằng sau các ViT thực tế: chúng cần hoặc nhiều dữ liệu huấn luyện hơn hẳn, hoặc tăng cường dữ liệu mạnh, hoặc một kiến trúc lai đưa trở lại một phần cấu trúc tích chập, để bắt kịp hiệu quả dữ liệu của CNN trên các tập dữ liệu nhỏ đến trung bình — vì tính bất biến tịnh tiến của CNN là một sự đảm bảo cứng về kiến trúc, trong khi của ViT phải được học từ các mẫu. Ở quy mô hàng trăm triệu ảnh, ViT bắt kịp hoặc vượt CNN một cách dễ dàng; ở quy mô hàng trăm ảnh, thì không.

### Bài tập

1. Tăng kích thước tập huấn luyện trong `make_dataset(data_rng, 300, ...)` từ 300 lên 3000 ảnh ở cùng các vị trí huấn luyện. Độ chính xác trên vị trí chưa thấy của ViT có cải thiện đáng kể, một phần, hay hoàn toàn không — và điều đó gợi ý gì về việc ViT cần thêm *bao nhiêu* dữ liệu để bù đắp cho thiên kiến quy nạp còn thiếu của nó?
2. Giảm `patch_size` từ 4 xuống 2 (mảng mịn hơn, chuỗi dài hơn). Chia mảng mịn hơn có giúp ViT tổng quát hóa tốt hơn sang các vị trí chưa thấy không, hay kiểu thất bại này không liên quan đến độ phân giải mảng?
3. Một kiến trúc "lai (hybrid)" đưa bản đồ đặc trưng của một CNN nhỏ (thay vì các mảng điểm ảnh thô) vào một bộ mã hóa Transformer, lấy lại được một phần thiên kiến quy nạp không gian của CNN. Hãy phác thảo cách bạn sửa `TinyViT.forward` để chia mảng *đầu ra* của một chồng `nn.Conv2d` nhỏ thay vì ảnh thô.